# Kaggle initial scaffold, restoration and backup history

These are original, unmodified Python source cells from the exported Kaggle notebook, organized by research phase. Execution outputs and counts have been cleared. Cells are in original index order within this phase. The Kaggle project workspace and some earlier phases are required; this notebook is not guaranteed to execute by itself from a clean environment. The original index for every cell is stored in cell metadata.

The historical record includes exploratory retries and some superseded checks. Do not rerun an early output-producing cell over frozen results without backing up the project first.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

from pathlib import Path
import hashlib
import zipfile

INPUT = Path("/kaggle/input")

print("=== STEP 3.1: BACKUP DISCOVERY ===")

print("\nAvailable Kaggle datasets:")
for folder in sorted(INPUT.iterdir()):
    print(" -", folder.name)

print("\n=== STEP 2 BACKUP CANDIDATES ===")

candidates = sorted(
    p for p in INPUT.rglob("*")
    if p.is_file()
    and (
        "asr_fluency_step02_complete" in p.name.lower()
        or p.name.lower() == "research_progress.json"
    )
)

for p in candidates:
    print("Path:", p)
    print("Size:", p.stat().st_size, "bytes")

    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p) as z:
            bad_file = z.testzip()
            print("ZIP integrity:",
                  "PASS" if bad_file is None else f"FAIL: {bad_file}")
            print("Archive entries:", len(z.namelist()))
            print(
                "Checkpoint exists:",
                "checkpoints/research_progress.json" in z.namelist()
            )

        sha = hashlib.sha256(p.read_bytes()).hexdigest()
        print("SHA256:", sha)

    print("-" * 50)

print("\nTotal candidates:", len(candidates))


In [ ]:

from pathlib import Path
import shutil
import json
import pandas as pd

SOURCE = Path(
    "/kaggle/input/datasets/abdulatifsafarov/"
    "asr-fluency-step02-complete"
)

PROJECT = Path(
    "/kaggle/working/acoustic_linguistic_research"
)

assert SOURCE.exists(), "Kaggle dataset not found"

required = [
    "checkpoints/research_progress.json",
    "configs/pilot_acoustic_v1.json",
    "data/metadata/pilot_selection_400.csv",
    "features/acoustic/pilot_acoustic_features_400.csv",
    "reports/step02_merged_validation.json"
]

print("=== STEP 3.1: DATASET RECOVERY ===")

for relative in required:
    assert (SOURCE / relative).is_file(), (
        f"Missing source file: {relative}"
    )

copied = 0
verified = 0

for src in sorted(SOURCE.rglob("*")):
    if not src.is_file():
        continue

    relative = src.relative_to(SOURCE)
    dst = PROJECT / relative
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        import hashlib
        def sha256(p):
            return hashlib.sha256(p.read_bytes()).hexdigest()

        assert sha256(src) == sha256(dst), (
            f"Existing file differs: {relative}"
        )
        verified += 1
    else:
        shutil.copy2(src, dst)
        copied += 1

checkpoint = json.loads(
    (PROJECT / "checkpoints/research_progress.json")
    .read_text(encoding="utf-8")
)

config = json.loads(
    (PROJECT / "configs/pilot_acoustic_v1.json")
    .read_text(encoding="utf-8")
)

validation = json.loads(
    (PROJECT / "reports/step02_merged_validation.json")
    .read_text(encoding="utf-8")
)

df = pd.read_csv(
    PROJECT / "features/acoustic/pilot_acoustic_features_400.csv",
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 400
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert len(config["features"]) == 14
assert all(col in df.columns for col in config["features"])
assert df["pitch_mean_hz"].isna().sum() == 2
assert validation["status"] == "PASS"
assert 2 in checkpoint["completed_steps"]

print("Files copied:", copied)
print("Already existing identical files:", verified)
print("Audio records:", len(df))
print("Speakers:", df["speaker"].nunique())
print("Acoustic features:", len(config["features"]))
print("Missing pitch:", df["pitch_mean_hz"].isna().sum())
print("Completed steps:", checkpoint["completed_steps"])
print("Step 2 validation:", validation["status"])
print("Project restored to:", PROJECT)
print("=== RECOVERY PASS ===")


In [ ]:

from pathlib import Path
import torch
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

expected = PROJECT / "data/raw/speechocean762_train.parquet"

print("=== STEP 3.1: AUDIO DATASET CHECK ===")

print("\nOriginal train Parquet exists:", expected.exists())

if expected.exists():
    print("Path:", expected)
    print("Size (MB):", round(expected.stat().st_size / 1048576, 2))

    pf = pq.ParquetFile(expected)
    print("Total rows:", pf.metadata.num_rows)
    print("Row groups:", pf.metadata.num_row_groups)
    print("Columns:", pf.schema.names)

    assert pf.metadata.num_rows == 2500
    assert "audio" in pf.schema.names

    print("Audio dataset: READY")
else:
    print("Original audio dataset: MISSING")
    print("Audio restoration will be required.")

print("\n=== GPU STATUS ===")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("\n=== OTHER PARQUET FILES ===")
for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    matches = list(root.rglob("*.parquet"))
    print(f"{root}: {len(matches)} parquet file(s)")
    for p in matches[:10]:
        print(" -", p)

print("\n=== AUDIO CHECK COMPLETED ===")


In [ ]:

import sys
import subprocess
import shutil
import hashlib
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q", "huggingface_hub"
    ])
    from huggingface_hub import HfApi, hf_hub_download

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
RAW = PROJECT / "data/raw"
RAW.mkdir(parents=True, exist_ok=True)

TARGET = RAW / "speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"

REPO = "mispeech/speechocean762"
REVISION = "06385584fad212b26134c656fdd3ccf9f093f33e"

print("=== STEP 3.1: TRAIN AUDIO RESTORATION ===")

if TARGET.exists():
    print("Existing Parquet found; validating it.")
else:
    api = HfApi()
    info = api.dataset_info(repo_id=REPO, revision=REVISION)

    parquet_files = [
        item.rfilename
        for item in info.siblings
        if item.rfilename.endswith(".parquet")
    ]

    train_files = [
        name for name in parquet_files
        if (
            name.startswith("data/train-")
            or name.startswith("train/")
            or Path(name).name.startswith("train-")
        )
    ]

    print("Train Parquet candidates:", train_files)

    assert len(train_files) == 1, (
        "Expected exactly one train Parquet. "
        "Inspect repository file list before continuing."
    )

    source = hf_hub_download(
        repo_id=REPO,
        repo_type="dataset",
        filename=train_files[0],
        revision=REVISION
    )

    shutil.copy2(source, TARGET)

pf = pq.ParquetFile(TARGET)

assert pf.metadata.num_rows == 2500, (
    f"Unexpected row count: {pf.metadata.num_rows}"
)
assert "audio" in pf.schema.names

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

expected_ids = set(pilot["audio_id"])
found_ids = set()

for group_id in range(pf.metadata.num_row_groups):
    audio_col = pf.read_row_group(
        group_id,
        columns=["audio"]
    ).column("audio").to_pylist()

    for audio in audio_col:
        audio_id = Path(audio["path"]).stem
        if audio_id in expected_ids:
            found_ids.add(audio_id)

assert len(expected_ids) == 400
assert found_ids == expected_ids, (
    f"Missing pilot IDs: {sorted(expected_ids-found_ids)[:10]}"
)

sha = hashlib.sha256()
with TARGET.open("rb") as f:
    for chunk in iter(lambda: f.read(1024*1024), b""):
        sha.update(chunk)

print("\n=== TRAIN AUDIO RESTORATION RESULT ===")
print("Path:", TARGET)
print("Size (MB):", round(TARGET.stat().st_size / 1048576, 2))
print("Train rows:", pf.metadata.num_rows)
print("Pilot audio IDs found:", len(found_ids))
print("Pilot audio IDs missing:", len(expected_ids-found_ids))
print("SHA256:", sha.hexdigest())
print("Status: PASS")


In [ ]:

from pathlib import Path
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")
TARGET = PROJECT / "data/raw/speechocean762_train.parquet"

assert TARGET.exists(), "Train Parquet file not found"

pf = pq.ParquetFile(TARGET)

print("=== PARQUET SCHEMA DIAGNOSTICS ===")

print("\nFile:", TARGET)
print("Size (MB):", round(TARGET.stat().st_size / 1048576, 2))
print("Rows:", pf.metadata.num_rows)
print("Row groups:", pf.metadata.num_row_groups)

print("\n=== PARQUET LEAF COLUMN NAMES ===")
print(pf.schema.names)

print("\n=== ARROW TOP-LEVEL COLUMN NAMES ===")
print(pf.schema_arrow.names)

print("\n=== FULL ARROW SCHEMA ===")
print(pf.schema_arrow)

print("\n=== AUDIO COLUMN CHECK ===")

if "audio" in pf.schema_arrow.names:
    print("Audio top-level column: FOUND")

    audio_field = pf.schema_arrow.field("audio")
    print("Audio field type:", audio_field.type)

    sample = pf.read_row_group(
        0, columns=["audio"]
    ).column("audio").slice(0, 1).to_pylist()

    if sample:
        print("First audio value type:", type(sample[0]).__name__)

        if isinstance(sample[0], dict):
            print("Audio struct keys:", list(sample[0].keys()))
            print("Audio path:", sample[0].get("path"))

            audio_bytes = sample[0].get("bytes")
            print(
                "Embedded audio bytes:",
                len(audio_bytes) if audio_bytes is not None else None
            )
else:
    print("Audio top-level column: NOT FOUND")
    print("Further schema investigation required.")

print("\n=== SCHEMA DIAGNOSTICS COMPLETE ===")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import pyarrow.parquet as pq

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

TRAIN = PROJECT / "data/raw/speechocean762_train.parquet"
PILOT = PROJECT / "data/metadata/pilot_selection_400.csv"
CHECK = PROJECT / "checkpoints/research_progress.json"
OUT = PROJECT / "reports/step03_audio_restoration_validation.json"

EXPECTED_SHA256 = (
    "b39d1a49b7279c7f5f0c1e7efaa72a20"
    "d1cde92738ac67cf84beabf756fe1901"
)

pilot = pd.read_csv(
    PILOT,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(pilot) == 400
assert pilot["audio_id"].is_unique
assert pilot["pilot_index"].tolist() == list(range(1, 401))

pf = pq.ParquetFile(TRAIN)

assert pf.metadata.num_rows == 2500
assert "audio" in pf.schema_arrow.names

expected = pilot.set_index("audio_id")
found = {}

for group_id in range(pf.metadata.num_row_groups):

    table = pf.read_row_group(
        group_id,
        columns=["audio", "speaker", "fluency"]
    )

    audio_values = table.column("audio").to_pylist()
    speakers = table.column("speaker").to_pylist()
    fluencies = table.column("fluency").to_pylist()

    for audio, speaker, fluency in zip(
        audio_values, speakers, fluencies
    ):
        audio_id = Path(audio["path"]).stem

        if audio_id in expected.index:

            assert audio_id not in found, (
                f"Duplicate audio ID: {audio_id}"
            )
            assert audio["bytes"] is not None

            found[audio_id] = {
                "speaker": str(speaker),
                "fluency": int(fluency),
                "audio_bytes": len(audio["bytes"])
            }

expected_ids = set(expected.index)
found_ids = set(found)

assert found_ids == expected_ids, (
    f"Missing audio IDs: {sorted(expected_ids - found_ids)}"
)

speaker_mismatches = []
fluency_mismatches = []

for audio_id, record in found.items():
    metadata = expected.loc[audio_id]

    if record["speaker"] != str(metadata["speaker"]):
        speaker_mismatches.append(audio_id)

    if record["fluency"] != int(metadata["fluency"]):
        fluency_mismatches.append(audio_id)

assert not speaker_mismatches
assert not fluency_mismatches

sha = hashlib.sha256()

with TRAIN.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha.update(chunk)

actual_sha256 = sha.hexdigest()

report = {
    "dataset": "mispeech/speechocean762",
    "revision": "06385584fad212b26134c656fdd3ccf9f093f33e",
    "train_rows": 2500,
    "pilot_audio_count": len(found),
    "missing_audio_ids": [],
    "speaker_mismatches": speaker_mismatches,
    "fluency_mismatches": fluency_mismatches,
    "train_parquet_sha256": actual_sha256,
    "matches_previous_sha256": (
        actual_sha256 == EXPECTED_SHA256
    ),
    "status": "PASS",
    "checked_at": datetime.now(timezone.utc).isoformat()
}

OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step3_audio_restoration"] = "PASS"
progress["step3_audio_restoration_report"] = str(OUT)
progress["next_task"] = "Design ASR hypothesis experiment"
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 3.1 AUDIO RESTORATION VALIDATION ===")
print("Train recordings:", pf.metadata.num_rows)
print("Pilot recordings matched:", len(found))
print("Missing pilot IDs:", len(expected_ids - found_ids))
print("Speaker mismatches:", len(speaker_mismatches))
print("Fluency mismatches:", len(fluency_mismatches))
print("Parquet SHA256:", actual_sha256)
print("Previous SHA256 match:",
      report["matches_previous_sha256"])
print("Status:", report["status"])
print("Report:", OUT)
print("Checkpoint updated.")


In [ ]:

import sys
import importlib.metadata as metadata

import torch
import pandas as pd

from pathlib import Path

PROJECT = Path("/kaggle/working/acoustic_linguistic_research")

print("=== STEP 3.2: ASR ENVIRONMENT DIAGNOSTICS ===")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)

        print(
            f"GPU {i}: {props.name}, "
            f"VRAM {props.total_memory / 1024**3:.2f} GiB"
        )

packages = [
    "transformers",
    "accelerate",
    "datasets",
    "openai-whisper",
    "faster-whisper",
    "ctranslate2",
    "librosa",
    "soundfile",
    "huggingface-hub"
]

print("\n=== PACKAGE VERSIONS ===")

for name in packages:
    try:
        version = metadata.version(name)
    except metadata.PackageNotFoundError:
        version = "NOT INSTALLED"

    print(f"{name}: {version}")

print("\n=== RESEARCH FILES ===")

files = [
    "features/acoustic/pilot_acoustic_features_400.csv",
    "data/metadata/pilot_selection_400.csv",
    "data/raw/speechocean762_train.parquet",
    "checkpoints/research_progress.json"
]

for relative in files:
    path = PROJECT / relative
    print(relative, ":", "FOUND" if path.exists() else "MISSING")

print("\n=== ASR ENVIRONMENT CHECK COMPLETED ===")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile
import os

ROOT = Path("/kaggle/working/acoustic_linguistic_research")
WORK = Path("/kaggle/working")
ZIP = WORK / "asr_fluency_step01-05_complete_with_step06_bootstrap.zip"

assert ROOT.is_dir(), "Research project directory missing"

EXPECTED_CRITICAL = {
    "data/modeling/modeling_dataset_0001-0400.csv":
        "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    "data/splits/speaker_groupkfold5_frozen.csv":
        "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    "configs/speaker_disjoint_cv_v1.json":
        "302c7ea0086a54d644a4ebc0b8a5d96d63575a3896b5a29d76dba7e26dacd472",
    "features/asr/full/asr_hypotheses_0001-0400.csv":
        "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80",
    "results/step05/baseline_acoustic_oof_predictions.csv":
        "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
    "results/step05/b1_transcript_ridge_oof_predictions.csv":
        "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    "results/step05/b2_prompt_aware_ridge_oof_predictions.csv":
        "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
    "results/step05/c1_fusion_ridge_oof_predictions.csv":
        "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv":
        "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9",
    "results/step05/speaker_cluster_bootstrap_10000_ci.csv":
        "5ef3d6dd01eb05758ccb8b4f77293396a81e988b9b9e33152cb666c6b48d1d93"
}

def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("=== VERIFY CRITICAL RESEARCH FILES ===")

for rel, expected in EXPECTED_CRITICAL.items():
    path = ROOT / rel
    assert path.is_file(), f"CRITICAL FILE MISSING: {rel}"
    assert digest(path) == expected, f"SHA MISMATCH: {rel}"
    print("PASS:", rel)

MAX_RAW_PARQUET_BYTES = 250 * 1024 * 1024
INCLUDED = []
OMITTED = []

for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or path.is_symlink():
        continue

    rel = path.relative_to(ROOT)
    rel_str = rel.as_posix()
    size = path.stat().st_size

    # Never package sealed official test data.
    if (
        "speechocean762_test" in rel_str.lower()
        or "official_test" in rel_str.lower()
    ):
        OMITTED.append((path, "sealed official test excluded"))
        continue

    # Raw parquet can be large; keep its SHA even if omitted.
    if (
        rel_str.startswith("data/raw/")
        and path.suffix.lower() == ".parquet"
        and size > MAX_RAW_PARQUET_BYTES
    ):
        OMITTED.append((path, "raw parquet exceeds 250 MB"))
        continue

    # Exclude temporary/cache files, not research deliverables.
    if (
        "__pycache__" in path.parts
        or ".ipynb_checkpoints" in path.parts
        or path.suffix in [".tmp", ".pyc"]
    ):
        OMITTED.append((path, "temporary or cache file"))
        continue

    INCLUDED.append((path, f"project/{rel_str}"))

# Add STEP 3 complete checkpoint, if available.
STEP3 = WORK / "asr_fluency_step03_complete.zip"
if STEP3.is_file():
    INCLUDED.append(
        (STEP3, "extra/asr_fluency_step03_complete.zip")
    )

# Package research source code/notebooks if saved separately.
for path in sorted(WORK.iterdir()):
    if (
        path.is_file()
        and path.suffix.lower() in [".py", ".ipynb", ".r", ".R"]
        and path != ZIP
    ):
        arc = f"extra/source_files/{path.name}"
        if arc not in [a for _, a in INCLUDED]:
            INCLUDED.append((path, arc))

assert INCLUDED
assert len({arc for _, arc in INCLUDED}) == len(INCLUDED)

entries = []
for path, arc in INCLUDED:
    entries.append({
        "archive_path": arc,
        "size_bytes": path.stat().st_size,
        "sha256": digest(path)
    })

omitted_entries = []
for path, reason in OMITTED:
    omitted_entries.append({
        "original_path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": digest(path),
        "reason": reason
    })

manifest = {
    "archive_purpose": "Restore PhD ASR fluency research through STEP 5",
    "step05_status": "COMPLETE",
    "step06_1_bootstrap_status": "COMPLETE",
    "step06_2_sensitivity_status": "NOT YET CONFIRMED",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "mispeech/speechocean762",
    "dataset_revision": "06385584fad212b26134c656fdd3ccf9f093f33e",
    "asr_model": "openai/whisper-small.en",
    "asr_model_revision": "e8727524f962ee844a7319d92be39ac1bd25655a",
    "unique_audio_count": 400,
    "speaker_count": 20,
    "asr_hypothesis_count": 2000,
    "cv": "frozen 5-fold speaker-disjoint",
    "ridge_alpha": 1.0,
    "official_test_used": False,
    "critical_file_hashes": EXPECTED_CRITICAL,
    "included_files": entries,
    "omitted_files": omitted_entries
}

restore = """ASR FLUENCY RESEARCH — RESTORE GUIDE

1. Upload this ZIP to a new Kaggle session.
2. Extract its project/ contents into:
   /kaggle/working/acoustic_linguistic_research/
3. Verify each restored file using MANIFEST.json SHA256 values.
4. STEP 5 is COMPLETE. Do not rerun its training cells.
5. STEP 6.1 bootstrap results are included.
6. Continue with STEP 6.2 anomaly sensitivity analysis.
7. Restore raw SpeechOcean762 train parquet separately if absent.
8. The official sealed test partition must remain untouched.
9. Original ASR model weights are not required to analyze
   already saved transcripts and OOF predictions.
10. Kaggle notebook cell history is not automatically captured
    by this archive unless a notebook/script exists as a file.
    Export the Kaggle notebook separately for full reproducibility.

IMPORTANT:
- Reading prompts are NOT verified ground-truth transcripts.
- Prompt-ASR disagreement is NOT WER or CER.
- All primary results retain the 400 selected audios.
- Sensitivity exclusions are diagnostic only.
- All predictions are from fixed-alpha Ridge (alpha=1.0).
- Review MANIFEST.json for any excluded large files.
"""

tmp_zip = ZIP.with_suffix(".zip.tmp")

with zipfile.ZipFile(
    tmp_zip, "w", compression=zipfile.ZIP_DEFLATED,
    compresslevel=6, allowZip64=True
) as z:
    for path, arc in INCLUDED:
        z.write(path, arcname=arc)

    z.writestr(
        "MANIFEST.json",
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )
    z.writestr("RESTORE_INSTRUCTIONS.txt", restore)

# Verify archive CRC and SHA256 for every included file.
with zipfile.ZipFile(tmp_zip, "r") as z:
    assert z.testzip() is None, "ZIP CRC ERROR"
    assert "MANIFEST.json" in z.namelist()
    assert "RESTORE_INSTRUCTIONS.txt" in z.namelist()

    for entry in entries:
        h = hashlib.sha256()
        with z.open(entry["archive_path"]) as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)
        assert h.hexdigest() == entry["sha256"], (
            "ZIP CONTENT HASH MISMATCH: "
            + entry["archive_path"]
        )

if ZIP.exists():
    assert digest(ZIP) == digest(tmp_zip), (
        "An archive with this name already exists and differs. "
        "Original archive preserved."
    )
    tmp_zip.unlink()
else:
    os.replace(tmp_zip, ZIP)

print("\n=== STEP 5 RESEARCH BACKUP COMPLETE ===")
print("Status: PASS")
print("STEP 5: COMPLETE")
print("STEP 6.1: COMPLETE")
print("Files included:", len(entries))
print("Files omitted:", len(omitted_entries))
print("STEP 3 archive included:", STEP3.is_file())
print("ZIP integrity: PASS")
print("All included SHA256 checks: PASS")
print("ZIP:", ZIP)
print("Size (MB):", round(ZIP.stat().st_size / 1024**2, 2))
print("ZIP SHA256:", digest(ZIP))

print("\n=== OMITTED FILES ===")
for item in omitted_entries:
    print(item["reason"], "|", item["original_path"])

print("\nNEXT: Download this ZIP and save it outside Kaggle.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import zipfile

ROOT = Path("/kaggle/working/acoustic_linguistic_research")
WORK = Path("/kaggle/working")
ARCHIVE = WORK / "asr_fluency_step01-06_complete.zip"

REQUIRED = [
    "data/modeling/modeling_dataset_0001-0400.csv",
    "data/splits/speaker_groupkfold5_frozen.csv",
    "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv",
    "results/step05/speaker_cluster_bootstrap_10000_ci.csv",
    "results/step05/step06_anomaly_sensitivity_400_399_396.csv",
    "results/step06_final_tables/table01_primary_models_with_ci.csv",
    "results/step06_final_tables/table02_paired_differences_with_ci.csv",
    "results/step06_final_tables/step06_variable_subset_bootstrap.csv",
    "results/step06_final_tables/step06_c2_critical_h2_vs_h1_contributions.csv",
    "results/step06_final_tables/step06_c2_coefficient_stability.csv",
    "results/step06_final_tables/step06_ood_diagnostic_per_hypothesis.csv",
    "results/step06_final_tables/step06_ood_diagnostic_summary.csv",
    "reports/step06_ood_feature_audit_v1.json",
]

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert ROOT.is_dir()

for rel in REQUIRED:
    assert (ROOT / rel).is_file(), f"Required file missing: {rel}"

# Verify previously frozen reference files.
KNOWN_HASHES = {
    "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv":
        "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9",
    "results/step05/speaker_cluster_bootstrap_10000_ci.csv":
        "5ef3d6dd01eb05758ccb8b4f77293396a81e988b9b9e33152cb666c6b48d1d93",
    "results/step05/step06_anomaly_sensitivity_400_399_396.csv":
        "83b34da4aa34634319b59c94391967565e3f3c0fd438fb5a80d48131ca5563d3",
    "results/step06_final_tables/table01_primary_models_with_ci.csv":
        "22512d4fdd54feec3f6658f99248653e8c9fb99e110887db64578aadfb43b4c0",
    "results/step06_final_tables/step06_variable_subset_bootstrap.csv":
        "257f53dd5eb50979bf4f3cb7a75de48be361baff049239e650c20f38f3d7bab9"
}

for rel, expected in KNOWN_HASHES.items():
    assert sha256(ROOT / rel) == expected, f"SHA mismatch: {rel}"

included = []
omitted = []

for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or path.is_symlink():
        continue

    rel = path.relative_to(ROOT).as_posix()

    if "official_test" in rel.lower() or "speechocean762_test" in rel.lower():
        omitted.append((path, "Sealed official test excluded"))
        continue

    if rel.startswith("data/raw/") and path.suffix == ".parquet":
        omitted.append((path, "Raw parquet stored externally"))
        continue

    if "__pycache__" in path.parts or path.suffix in [".tmp", ".pyc"]:
        continue

    included.append((path, "project/" + rel))

# Include the previous complete STEP 5 checkpoint.
OLD_BACKUP = WORK / "asr_fluency_step01-05_complete_with_step06_bootstrap.zip"
if OLD_BACKUP.is_file():
    included.append((OLD_BACKUP, "extra/step01-05_previous_checkpoint.zip"))

STEP3 = WORK / "asr_fluency_step03_complete.zip"
if STEP3.is_file():
    included.append((STEP3, "extra/asr_fluency_step03_complete.zip"))

# Include saved notebooks/scripts if present.
for path in sorted(WORK.iterdir()):
    if path.is_file() and path.suffix.lower() in [".py", ".ipynb", ".r"]:
        included.append((path, "extra/source_files/" + path.name))

assert len({name for _, name in included}) == len(included)

entries = [
    {
        "archive_path": name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path)
    }
    for path, name in included
]

manifest = {
    "project": "ASR fluency prediction accuracy and stability",
    "status": "STEP_1_TO_6_COMPLETE",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "mispeech/speechocean762",
    "dataset_revision": "06385584fad212b26134c656fdd3ccf9f093f33e",
    "audio_count": 400,
    "speaker_count": 20,
    "hypothesis_count": 2000,
    "cv": "5-fold speaker-disjoint",
    "models": ["M0", "A", "B1", "B2", "C1", "C2"],
    "analysis_complete": [
        "Fixed-alpha Ridge evaluation",
        "Speaker-clustered bootstrap",
        "ASR anomaly sensitivity",
        "Integrated research tables",
        "ASR variability association",
        "Variable-subset bootstrap",
        "Critical feature contribution",
        "Coefficient stability and multicollinearity",
        "Feature extremeness audit"
    ],
    "official_test_untouched": True,
    "included_files": entries,
    "omitted_files": [
        {
            "original_path": str(path),
            "reason": reason,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path)
        }
        for path, reason in omitted
    ],
    "research_cautions": [
        "Reading prompt is not a verified transcription",
        "Prompt-ASR disagreement is not ground-truth WER/CER",
        "Only 20 independent speaker clusters",
        "Hypotheses from the same audio are correlated",
        "Bootstrap does not refit models",
        "Sensitivity analyses do not replace the primary 400-audio analysis",
        "Extreme feature threshold is exploratory"
    ]
}

restore = """RESTORE STEP 1-6 RESEARCH

1. Upload this ZIP to Kaggle.
2. Extract project/ files to:
   /kaggle/working/acoustic_linguistic_research/
3. Check MANIFEST.json for file hashes and omitted files.
4. STEP 1-6 experiments are complete.
5. Do not retrain the fixed-alpha Ridge models unnecessarily.
6. Keep the official test partition sealed.
7. Continue with manuscript preparation, further validation
   planning, or reviewer-requested analysis.
8. Export the Kaggle notebook separately as .ipynb.
9. The original train Parquet is excluded; retrieve the
   pinned train revision if raw audio is needed.
"""

temp = WORK / (ARCHIVE.name + ".tmp")

with zipfile.ZipFile(
    temp, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True
) as z:
    for path, name in included:
        z.write(path, arcname=name)

    z.writestr(
        "MANIFEST.json",
        json.dumps(manifest, indent=2, ensure_ascii=False)
    )
    z.writestr("RESTORE_INSTRUCTIONS.txt", restore)

with zipfile.ZipFile(temp) as z:
    assert z.testzip() is None

    for entry in entries:
        h = hashlib.sha256()
        with z.open(entry["archive_path"]) as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)

        assert h.hexdigest() == entry["sha256"], (
            f"Archive hash mismatch: {entry['archive_path']}"
        )

if ARCHIVE.exists():
    # A completed earlier backup is preserved.
    assert sha256(ARCHIVE) == sha256(temp), (
        "Archive already exists with different content. "
        "Original preserved; rename the new backup before saving."
    )
    temp.unlink()
else:
    os.replace(temp, ARCHIVE)

print("=== STEP 6 FINAL CHECKPOINT ===")
print("Status: PASS")
print("STEP 5: COMPLETE")
print("STEP 6: COMPLETE")
print("Included files:", len(entries))
print("Omitted files:", len(omitted))
print("ZIP integrity: PASS")
print("SHA256 verification: PASS")
print("Archive:", ARCHIVE)
print("Size MB:", round(ARCHIVE.stat().st_size / 1024**2, 2))
print("SHA256:", sha256(ARCHIVE))

print("\n=== OMITTED FILES ===")
for path, reason in omitted:
    print(reason, "|", path)

print("\nNEXT: Download ZIP and export Kaggle notebook.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import zipfile

ROOT = Path("/kaggle/working/acoustic_linguistic_research")
WORK = Path("/kaggle/working")
ARCHIVE = WORK / "asr_fluency_step01-07_complete.zip"
TEMP = WORK / "asr_fluency_step01-07_complete.zip.tmp"

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

REQUIRED = {
    "results/step07_methodology/step07_01_dataset_integrity.json":
        "9b48a723f15fc080c86e23a9e3b61b6183917818de4badc79770080a46eb8a31",
    "results/step07_methodology/step07_02_leakage_audit.json":
        "d0deb85ae53f034664f870132913cea3498b26a22caed7c042a66e8a41d900d0",
    "results/step07_methodology/step07_03_regularization_fairness.json":
        "7a10748ff0e91cd199ea0b5242aae28da1f7f7aba5deeec03f86ae00b14a701f",
    "results/step07_methodology/step07_04_prompt_validity_audit.json":
        "2d1f90d7d6d87a82a181bba300c14bee14e9c468ce0dd98b5a37877ab3709326",
    "results/step07_methodology/step07_05_statistical_claims_audit.json":
        "f0b9f8aedcf1e72ea467cff21bde9d33836b43bde3c5a81cb8a0863f86ad2f8a",
    "results/step07_methodology/step07_06_reviewer_readiness.json":
        "a74ad6f9c3fd7b54ba92575c90914054e0ddd59079d30686f436c3989e1c6c6f",
    "results/step07_methodology/step07_06_reviewer_decision_registry.csv":
        "eb998dbe6a6757940985202166be4793480fd56851a014bb6b6deddf4eeed08b"
}

assert ROOT.is_dir()

print("=== VERIFY STEP 7 AUDITS ===")
for rel, expected in REQUIRED.items():
    path = ROOT / rel
    assert path.is_file(), f"Missing: {rel}"
    assert sha256(path) == expected, f"SHA mismatch: {rel}"
    print("PASS:", rel)

readiness_path = ROOT / (
    "results/step07_methodology/"
    "step07_06_reviewer_readiness.json"
)
readiness = json.loads(readiness_path.read_text(encoding="utf-8"))

assert readiness["status"] == "PASS"
assert readiness["manuscript_readiness"] == (
    "CONDITIONAL_NOT_SUBMISSION_READY"
)

items = []
omitted = []

for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or path.is_symlink():
        continue

    rel = path.relative_to(ROOT).as_posix()
    lower = rel.lower()

    if "official_test" in lower or "speechocean762_test" in lower:
        omitted.append((path, "Sealed official test excluded"))
        continue

    if rel.startswith("data/raw/") and path.suffix.lower() == ".parquet":
        omitted.append((path, "Original train parquet excluded"))
        continue

    if (
        "__pycache__" in path.parts
        or ".ipynb_checkpoints" in path.parts
        or path.suffix.lower() in [".tmp", ".pyc"]
    ):
        continue

    items.append((path, "project/" + rel))

PREVIOUS = WORK / "asr_fluency_step01-06_complete.zip"
if PREVIOUS.is_file():
    items.append(
        (PREVIOUS, "extra/asr_fluency_step01-06_complete.zip")
    )

# Include available notebooks/scripts from Kaggle working directory.
for path in sorted(WORK.iterdir()):
    if (
        path.is_file()
        and path.suffix.lower() in [".ipynb", ".py", ".r"]
    ):
        items.append((path, "extra/source_files/" + path.name))

assert len(items) > 0
assert len(set(arc for _, arc in items)) == len(items)

entries = [
    {
        "archive_path": arc,
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path)
    }
    for path, arc in items
]

manifest = {
    "project": "ASR fluency accuracy and stability research",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "completed_steps": [1, 2, 3, 4, 5, 6, 7],
    "step07_audits": "PASS",
    "manuscript_readiness": "CONDITIONAL_NOT_SUBMISSION_READY",
    "unresolved_items": readiness["unresolved_or_limited_items"],
    "dataset": "mispeech/speechocean762",
    "dataset_revision":
        "06385584fad212b26134c656fdd3ccf9f093f33e",
    "original_audios": 400,
    "speakers": 20,
    "asr_hypotheses": 2000,
    "primary_results_unchanged": True,
    "official_test_partition": "not evaluated",
    "included_files": entries,
    "omitted_files": [
        {
            "original_path": str(path),
            "reason": reason,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path)
        }
        for path, reason in omitted
    ]
}

instructions = """
RESTORE INSTRUCTIONS — STEP 1 TO STEP 7

1. Upload this ZIP to a Kaggle session.
2. Extract project/ into:
   /kaggle/working/acoustic_linguistic_research/
3. Verify SHA256 values using MANIFEST.json.
4. STEP 1-7 calculations and audits are completed.
5. Do not automatically retrain or overwrite original models.
6. Review D07 and D08 before final manuscript Methods.
7. Official test partition remains sealed.
8. The original train parquet is excluded.
9. Export the original Kaggle notebook separately as .ipynb.
10. Continue with provenance verification and STEP 8 figures.
"""

# If already created, validate instead of silently overwriting.
if ARCHIVE.exists():
    with zipfile.ZipFile(ARCHIVE, "r") as z:
        assert z.testzip() is None
        old_manifest = json.loads(
            z.read("MANIFEST.json").decode("utf-8")
        )

        required_archive_paths = {
            "project/" + rel for rel in REQUIRED
        }
        assert required_archive_paths.issubset(
            set(z.namelist())
        )

        for entry in old_manifest["included_files"]:
            h = hashlib.sha256()
            with z.open(entry["archive_path"]) as f:
                for block in iter(
                    lambda: f.read(1024 * 1024), b""
                ):
                    h.update(block)
            assert h.hexdigest() == entry["sha256"]

    print("\nExisting STEP 7 backup verified.")
else:
    with zipfile.ZipFile(
        TEMP, "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
        allowZip64=True
    ) as z:
        for path, arc in items:
            z.write(path, arcname=arc)

        z.writestr(
            "MANIFEST.json",
            json.dumps(manifest, indent=2, ensure_ascii=False)
        )
        z.writestr("RESTORE_INSTRUCTIONS.txt", instructions)

    with zipfile.ZipFile(TEMP, "r") as z:
        assert z.testzip() is None
        for entry in entries:
            h = hashlib.sha256()
            with z.open(entry["archive_path"]) as f:
                for block in iter(
                    lambda: f.read(1024 * 1024), b""
                ):
                    h.update(block)
            assert h.hexdigest() == entry["sha256"], (
                entry["archive_path"]
            )

    os.replace(TEMP, ARCHIVE)

with zipfile.ZipFile(ARCHIVE) as z:
    final_manifest = json.loads(
        z.read("MANIFEST.json").decode("utf-8")
    )
    file_count = len(final_manifest["included_files"])
    omitted_count = len(final_manifest["omitted_files"])

print("\n=== STEP 7 FINAL CHECKPOINT ===")
print("Status: PASS")
print("STEP 1-7: COMPLETE")
print("Manuscript readiness: CONDITIONAL")
print("Included files:", file_count)
print("Omitted files:", omitted_count)
print("ZIP integrity: PASS")
print("SHA256 verification: PASS")
print("Archive:", ARCHIVE)
print("Size MB:", round(ARCHIVE.stat().st_size / 1024**2, 2))
print("SHA256:", sha256(ARCHIVE))
print("Unresolved items:",
      final_manifest["unresolved_items"])
print("\nNEXT: Download ZIP and export Kaggle notebook.")


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import zipfile
import io

P = Path("/kaggle/working/acoustic_linguistic_research")
ROOT = Path("/kaggle/working")

ARCHIVE = ROOT / "asr_fluency_step08_publication_complete.zip"
MANIFEST_NAME = "step08_backup_manifest.json"

FOLDERS = [
    P / "results/step08_provenance",
    P / "results/step08_figures",
    P / "results/step08_publication_tables",
    P / "results/step08_publication_audit",
    P / "results/step06_final_tables"
]

EXTRA = [
    P / "configs/asr_hypothesis_pilot_v1.json",
    P / "configs/speaker_disjoint_cv_v1.json",
    P / "checkpoints/pilot_speaker_ids.json",
    P / "data/metadata/pilot_selection_400.csv"
]

def sha_bytes(content):
    return hashlib.sha256(content).hexdigest()

def sha_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

files = {}

for folder in FOLDERS:
    assert folder.is_dir(), f"Missing folder: {folder}"
    for path in folder.rglob("*"):
        if path.is_file():
            files[str(path.relative_to(P))] = path

for path in EXTRA:
    assert path.is_file(), f"Missing file: {path}"
    files[str(path.relative_to(P))] = path

assert len(files) > 30

# Mandatory completed artifacts.
required = [
    "results/step08_figures/figure01_research_workflow.png",
    "results/step08_figures/figure02_model_performance_95ci.png",
    "results/step08_figures/figure03_accuracy_stability_tradeoff.png",
    "results/step08_figures/figure04_lexical_variation_prediction_range.png",
    "results/step08_figures/figure05_critical_asr_hallucination.png",
    "results/step08_figures/figure05b_h2_h1_feature_contributions.png",
    "results/step08_publication_tables/table02a_model_performance.csv",
    "results/step08_publication_tables/table02b_paired_bootstrap_comparisons.csv",
    "results/step08_publication_tables/table03_anomaly_sensitivity_publication.csv",
    "results/step08_provenance/step08_09_final_provenance_record.json",
    "results/step08_publication_audit/step08_17_publication_artifact_audit.json"
]

missing = [name for name in required if name not in files]
assert not missing, f"Missing required artifacts: {missing}"

entries = []

for name, path in sorted(files.items()):
    entries.append({
        "path": name,
        "size_bytes": path.stat().st_size,
        "sha256": sha_file(path)
    })

manifest = {
    "archive_purpose": "STEP 8 publication outputs and provenance",
    "project": "Within-Utterance ASR Transcription Variability",
    "status": "READY_FOR_ARCHIVE",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "file_count": len(entries),
    "files": entries,
    "exclusions": [
        "Raw audio and large source parquet files",
        "Full STEP 1-7 research backup (stored separately)",
        "Kaggle notebook source code (export separately)"
    ],
    "important_notes": [
        "Primary modeling results are frozen.",
        "STEP 8 only generated publication outputs and audits.",
        "Prompt-aware models are secondary ablations.",
        "No external test-split validation was performed.",
        "Original ASR runtime was not independently replayed."
    ]
}

manifest_bytes = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False
).encode("utf-8")

# Write to temporary archive before replacing final path.
TEMP = ROOT / "asr_fluency_step08_publication_complete.tmp.zip"

with zipfile.ZipFile(
    TEMP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as z:
    for name, path in sorted(files.items()):
        z.write(path, arcname=name)

    z.writestr(MANIFEST_NAME, manifest_bytes)

# Verify archive against the independent file manifest.
with zipfile.ZipFile(TEMP, "r") as z:
    assert z.testzip() is None

    names = set(z.namelist())
    assert len(names) == len(entries) + 1
    assert MANIFEST_NAME in names

    restored_manifest = json.loads(
        z.read(MANIFEST_NAME).decode("utf-8")
    )

    assert restored_manifest["file_count"] == len(entries)

    for entry in restored_manifest["files"]:
        archived = z.read(entry["path"])
        assert len(archived) == entry["size_bytes"]
        assert sha_bytes(archived) == entry["sha256"]

TEMP.replace(ARCHIVE)

assert ARCHIVE.is_file()

# Verify final archive is readable.
with zipfile.ZipFile(ARCHIVE, "r") as z:
    assert z.testzip() is None

print("=== STEP 8.19 PUBLICATION BACKUP ===")
print("Status: VERIFIED")
print("Files archived:", len(entries))
print("ZIP entries including manifest:", len(entries) + 1)
print("Required publication artifacts:", len(required))
print("All archive file hashes: PASS")

print("\n=== ARCHIVE ===")
print("Path:", ARCHIVE)
print("Size MB:", round(ARCHIVE.stat().st_size / (1024**2), 2))
print("SHA256:", sha_file(ARCHIVE))

print("\n=== PRESERVATION ===")
print("STEP 1-7 archive modified: NO")
print("Original modeling datasets modified: NO")
print("Raw audio included: NO")
print("Kaggle notebook included: NO")
print("Next: download ZIP and export notebook separately.")
